# 02 — Batch Sound-Facet Embedding (Day 2, Core tier)

Runs the real `sonic_explorer` package (cloned from GitHub, not reimplemented here)
against the curated FMA library sitting in Drive: segments every track, embeds every
segment with CLAP, and stores everything in a SQLite DB + FAISS index -- also in
Drive, so a Colab disconnect never loses completed work.

**This is inference only, not training.** CLAP is used exactly as pretrained
(`laion/clap-htsat-unfused`) -- no fine-tuning happens in Core tier.

**Requires a GPU runtime**: Runtime -> Change runtime type -> T4 GPU (or better).
Unlike notebook 01, this step is meaningfully faster with one.

**Output** (in `MyDrive/SonicExplorer/artifacts/`):
- `sonic_explorer.db` -- songs, segments, embedding_status
- `sound.index` -- FAISS index of every segment's CLAP embedding

These are the small artifacts that get synced down to the local repo's `data/artifacts/`
for local Streamlit development -- the curated audio gets synced separately.

## 1. Clone the repo and install sonic_explorer

Same package as local dev -- there is exactly one implementation of `Song`,
`Segment`, `Facet`, `FacetRegistry`, the repositories, etc. If a bug gets found and
fixed here, the fix belongs in the repo (commit + push), never only in this notebook.

In [ ]:
import os
import subprocess
import sys

REPO_URL = 'https://github.com/oyoai/sonic-explorer.git'
REPO_DIR = '/content/sonic-explorer'


def run(cmd):
    print('$', ' '.join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f'Command failed (exit {result.returncode}): {" ".join(cmd)}')


if os.path.exists(f'{REPO_DIR}/.git'):
    run(['git', '-C', REPO_DIR, 'pull'])
else:
    run(['git', 'clone', REPO_URL, REPO_DIR])

run([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{REPO_DIR}[colab]'])

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print('sonic_explorer installed from', REPO_DIR)

## 2. Mount Drive and set up paths

The DB and FAISS index live directly on Drive (not `/content`) for the whole run --
every `add_vector`/`mark_done` call commits straight to the Drive-mounted SQLite file,
and we checkpoint-save the FAISS index periodically below. If this runtime disconnects
partway through, re-running this notebook picks up exactly where it left off.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/SonicExplorer')
CURATED_DIR = DRIVE_ROOT / 'fma_curated'
ARTIFACTS_DIR = DRIVE_ROOT / 'artifacts'
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

DB_PATH = ARTIFACTS_DIR / 'sonic_explorer.db'

print('Curated audio:', CURATED_DIR)
print('Artifacts (DB + index):', ARTIFACTS_DIR)

## 3. Load the curated manifest (from notebook 01)

In [ ]:
import pandas as pd

manifest = pd.read_csv(CURATED_DIR / 'curated_tracks.csv')
print(f'{len(manifest)} curated tracks')
manifest.head()

## 4. Set up repositories + the sound facet

`embedding_repo.load_index('sound')` picks up any FAISS index already saved from a
prior (possibly disconnected) run of this notebook -- part of the same compute-once
pattern that `embedding_status` gives us at the segment level.

In [ ]:
from sonic_explorer.repository.db import init_db
from sonic_explorer.repository.song_repository import SongRepository
from sonic_explorer.repository.embedding_repository import EmbeddingRepository
from sonic_explorer.facets.sound import SoundFacet

conn = init_db(DB_PATH)
song_repo = SongRepository(conn)
embedding_repo = EmbeddingRepository(conn, artifacts_dir=ARTIFACTS_DIR)
embedding_repo.load_index('sound')

sound_facet = SoundFacet(use_clap=True)

print('Resuming with existing sound index size:', embedding_repo.index_size('sound'))

## 5. Run the batch pipeline

For each track: segment into 5s/2.5s-hop windows, CLAP-embed any segment not already
marked `'done'`, store the vectors. Already-fully-embedded tracks are skipped entirely
(no audio reload) -- this is what makes re-running after a disconnect cheap.

Checkpointing (every 50 tracks, and once more at the end) is handled *inside*
`run_batch_embedding` itself, not by this cell -- specifically so a segment is only
ever marked `'done'` in the DB *after* its vector has actually been saved to the
FAISS index on disk. An earlier version marked `'done'` immediately per-vector,
decoupled from when the index was next saved; a disconnect between those two
moments left a handful of segments claiming `'done'` with no vector actually
persisted -- silent data loss that only surfaced later as a crash. The callback
below is just for progress logging now.

In [ ]:
from sonic_explorer.pipeline.embed_library import run_batch_embedding

def log_progress(done, total):
    print(f'[{done}/{total}] checkpoint -- sound index now has {embedding_repo.index_size("sound")} vectors')

run_batch_embedding(
    manifest_df=manifest,
    audio_dir=CURATED_DIR,
    song_repo=song_repo,
    embedding_repo=embedding_repo,
    sound_facet=sound_facet,
    facet_name='sound',
    checkpoint_every=50,
    on_checkpoint=log_progress,
)

print('Batch embedding complete.')
print('Final sound index size:', embedding_repo.index_size('sound'))
print('Total songs in DB:', len(song_repo.list_songs()))

## 6. Sanity check -- does retrieval actually return something sensible?

Before syncing anything down, spot-check: pick a random song's middle segment, pull
its already-computed vector back out of the index (`get_vector`, no re-embedding
needed), and look at its nearest neighbors *in other songs*. Same genre showing up
repeatedly here is a good sign; if results look random, something's wrong before we
go further.

(Same-song neighbors are excluded deliberately -- overlapping 5s windows 2.5s apart
share literal audio content, so a song's own segments are close to guaranteed to
dominate an unfiltered top-k and would tell us nothing. This exclusion is exactly
what `RetrievalService.query_by_segment` -- what the real Moment Matcher UI calls --
does by default.)

In [ ]:
import random

sample_song = random.choice(song_repo.list_songs())
sample_song = song_repo.get_song(sample_song.id)
query_seg = sample_song.segments[len(sample_song.segments) // 2]
query_vec = embedding_repo.get_vector('sound', query_seg.id)

raw_matches = embedding_repo.search('sound', query_vec, k=20)
other_song_matches = [
    (seg_id, score) for seg_id, score in raw_matches
    if song_repo.get_segment(seg_id).song_id != sample_song.id
][:6]

print(f'Query: "{sample_song.title}" by {sample_song.artist} ({sample_song.genre_top}), '
      f'segment {query_seg.start_sec:.1f}-{query_seg.end_sec:.1f}s\n')
for seg_id, score in other_song_matches:
    seg = song_repo.get_segment(seg_id)
    match_song = song_repo.get_song(seg.song_id)
    print(f'  {score:.3f}  "{match_song.title}" by {match_song.artist} ({match_song.genre_top})')

## Done

`MyDrive/SonicExplorer/artifacts/sonic_explorer.db` and `sound.index` now hold the
full curated library's sound-facet embeddings. Next: sync these two (small) files
down to this repo's local `data/artifacts/` folder, plus the curated audio to
`data/audio/`, to start local `RetrievalService` + Streamlit development.